# Credit Risk Feature Engineering

## Objective

This notebook transforms the raw Home Credit application data into a modeling-ready dataset.

The objectives are to:

1. Split the data before learning preprocessing parameters
2. Correct invalid and special values
3. Create interpretable credit-risk features
4. Avoid duplicate information and unnecessary multicollinearity
5. Define numerical and categorical feature groups
6. Build a reproducible preprocessing workflow
7. Save the fitted preprocessing artifacts for baseline modeling

This notebook focuses on feature engineering and preprocessing. Model training and evaluation will be completed in the next notebook.

## Modeling Principles

- Split the data before fitting imputers, encoders, or scalers
- Use only information available at application time
- Preserve missingness when it may carry risk information
- Handle special values inside reusable transformation functions
- Handle division-by-zero explicitly
- Avoid retaining exact linear duplicates of engineered features
- Apply identical transformations to training and test data
- Fit all learned preprocessing parameters using training data only

## 1. Setup and Data Loading

In [97]:
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option("display.max_columns", 150)
pd.set_option("display.max_rows", 100)

print("Python version:", sys.version)

Python version: 3.13.1 (main, Dec  3 2024, 17:59:52) [Clang 16.0.0 (clang-1600.0.26.4)]


In [98]:
PROJECT_ROOT = Path.cwd().parent
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
MODEL_DIR = PROJECT_ROOT / "models"

TRAIN_PATH = RAW_DATA_DIR / "application_train.csv"

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(TRAIN_PATH)

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (307511, 122)


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,NAME_TYPE_SUITE,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,REGION_POPULATION_RELATIVE,DAYS_BIRTH,DAYS_EMPLOYED,DAYS_REGISTRATION,DAYS_ID_PUBLISH,OWN_CAR_AGE,FLAG_MOBIL,FLAG_EMP_PHONE,FLAG_WORK_PHONE,FLAG_CONT_MOBILE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,REGION_RATING_CLIENT,REGION_RATING_CLIENT_W_CITY,WEEKDAY_APPR_PROCESS_START,HOUR_APPR_PROCESS_START,REG_REGION_NOT_LIVE_REGION,REG_REGION_NOT_WORK_REGION,LIVE_REGION_NOT_WORK_REGION,REG_CITY_NOT_LIVE_CITY,REG_CITY_NOT_WORK_CITY,LIVE_CITY_NOT_WORK_CITY,ORGANIZATION_TYPE,EXT_SOURCE_1,EXT_SOURCE_2,EXT_SOURCE_3,APARTMENTS_AVG,BASEMENTAREA_AVG,YEARS_BEGINEXPLUATATION_AVG,YEARS_BUILD_AVG,COMMONAREA_AVG,ELEVATORS_AVG,ENTRANCES_AVG,FLOORSMAX_AVG,FLOORSMIN_AVG,LANDAREA_AVG,LIVINGAPARTMENTS_AVG,LIVINGAREA_AVG,NONLIVINGAPARTMENTS_AVG,NONLIVINGAREA_AVG,APARTMENTS_MODE,BASEMENTAREA_MODE,YEARS_BEGINEXPLUATATION_MODE,YEARS_BUILD_MODE,COMMONAREA_MODE,ELEVATORS_MODE,ENTRANCES_MODE,FLOORSMAX_MODE,FLOORSMIN_MODE,LANDAREA_MODE,LIVINGAPARTMENTS_MODE,LIVINGAREA_MODE,NONLIVINGAPARTMENTS_MODE,NONLIVINGAREA_MODE,APARTMENTS_MEDI,BASEMENTAREA_MEDI,YEARS_BEGINEXPLUATATION_MEDI,YEARS_BUILD_MEDI,COMMONAREA_MEDI,ELEVATORS_MEDI,ENTRANCES_MEDI,FLOORSMAX_MEDI,FLOORSMIN_MEDI,LANDAREA_MEDI,LIVINGAPARTMENTS_MEDI,LIVINGAREA_MEDI,NONLIVINGAPARTMENTS_MEDI,NONLIVINGAREA_MEDI,FONDKAPREMONT_MODE,HOUSETYPE_MODE,TOTALAREA_MODE,WALLSMATERIAL_MODE,EMERGENCYSTATE_MODE,OBS_30_CNT_SOCIAL_CIRCLE,DEF_30_CNT_SOCIAL_CIRCLE,OBS_60_CNT_SOCIAL_CIRCLE,DEF_60_CNT_SOCIAL_CIRCLE,DAYS_LAST_PHONE_CHANGE,FLAG_DOCUMENT_2,FLAG_DOCUMENT_3,FLAG_DOCUMENT_4,FLAG_DOCUMENT_5,FLAG_DOCUMENT_6,FLAG_DOCUMENT_7,FLAG_DOCUMENT_8,FLAG_DOCUMENT_9,FLAG_DOCUMENT_10,FLAG_DOCUMENT_11,FLAG_DOCUMENT_12,FLAG_DOCUMENT_13,FLAG_DOCUMENT_14,FLAG_DOCUMENT_15,FLAG_DOCUMENT_16,FLAG_DOCUMENT_17,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,351000.0,Unaccompanied,Working,Secondary / secondary special,Single / not married,House / apartment,0.018801,-9461,-637,-3648.0,-2120,NaN,1,1,0,1,1,0,Laborers,1.0,2,2,WEDNESDAY,10,0,0,0,0,0,0,Business Entity Type 3,0.083037,0.262949,0.139376,0.0247,0.0369,0.9722,0.6192,0.0143,0.00,0.0690,0.0833,0.1250,0.0369,0.0202,0.0190,0.0000,0.0000,0.0252,0.0383,0.9722,0.6341,0.0144,0.0000,0.0690,0.0833,0.1250,0.0377,0.022,0.0198,0.0,0.0,0.0250,0.0369,0.9722,0.6243,0.0144,0.00,0.0690,0.0833,0.1250,0.0375,0.0205,0.0193,0.0000,0.00,reg oper account,block of flats,0.0149,"Stone, brick",No,2.0,2.0,2.0,2.0,-1134.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,1129500.0,Family,State servant,Higher education,Married,House / apartment,0.003541,-16765,-1188,-1186.0,-291,NaN,1,1,0,1,1,0,Core staff,2.0,1,1,MONDAY,11,0,0,0,0,0,0,School,0.311267,0.622246,NaN,0.0959,0.0529,0.9851,0.7960,0.0605,0.08,0.0345,0.2917,0.3333,0.0130,0.0773,0.0549,0.0039,0.0098,0.0924,0.0538,0.9851,0.8040,0.0497,0.0806,0.0345,0.2917,0.3333,0.0128,0.079,0.0554,0.0,0.0,0.0968,0.0529,0.9851,0.7987,0.0608,0.08,0.0345,0.2917,0.3333,0.0132,0.0787,0.0558,0.0039,0.01,reg oper account,block of flats,0.0714,Block,No,1.0,0.0,1.0,0.0,-828.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,135000.0,Unaccompanied,Working,Secondary / secondary special,Single / not married,House / apartment,0.010032,-19046,-225,-4260.0,-2531,26.0,1,1,1,1,1,0,Laborers,1.0,2,2,MONDAY,9,0,0,0,0,0,0,Government,NaN,0.555912,0.729567,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,

## 2. Basic Validation

In [99]:
required_columns = ["SK_ID_CURR", "TARGET"]

missing_required_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_required_columns:
    raise ValueError(
        f"Missing required columns: {missing_required_columns}"
    )

print("Rows:", len(df))
print("Columns:", df.shape[1])
print("Unique applicants:", df["SK_ID_CURR"].nunique())
print("Duplicate applicant IDs:", df["SK_ID_CURR"].duplicated().sum())
print("\nTarget distribution:")
print(df["TARGET"].value_counts(dropna=False))
print("\nTarget rate:")
print(df["TARGET"].mean())

Rows: 307511
Columns: 122
Unique applicants: 307511
Duplicate applicant IDs: 0

Target distribution:
TARGET
0    282686
1     24825
Name: count, dtype: int64

Target rate:
0.08072881945686496


In [100]:
assert df["SK_ID_CURR"].duplicated().sum() == 0
assert set(df["TARGET"].dropna().unique()).issubset({0, 1})

### Observation

- The application table contains one row per applicant.
- `SK_ID_CURR` is an identifier and should not be used as a model feature.
- `TARGET = 1` represents applicants with repayment difficulties.
- The target is imbalanced, so accuracy alone will not be sufficient for model evaluation.

## 3. Train/Test Split

In [101]:
X = df.drop(columns=["TARGET"])
y = df["TARGET"].astype("int8")

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("Training target rate:", y_train.mean())
print("Test target rate:", y_test.mean())

X_train shape: (246008, 121)
X_test shape: (61503, 121)
Training target rate: 0.08072908198107379
Test target rate: 0.08072776937710356


In [102]:
assert len(X_train) + len(X_test) == len(df)
assert set(X_train.index).isdisjoint(set(X_test.index))
assert abs(y_train.mean() - y_test.mean()) < 0.001

In [103]:
train_ids = X_train["SK_ID_CURR"].copy()
test_ids = X_test["SK_ID_CURR"].copy()

X_train = X_train.drop(columns=["SK_ID_CURR"])
X_test = X_test.drop(columns=["SK_ID_CURR"])

## 4. Initial Feature Scope

In [104]:
initial_features = [
    "NAME_CONTRACT_TYPE",
    "CODE_GENDER",
    "FLAG_OWN_CAR",
    "FLAG_OWN_REALTY",
    "CNT_CHILDREN",
    "CNT_FAM_MEMBERS",
    "AMT_INCOME_TOTAL",
    "AMT_CREDIT",
    "AMT_ANNUITY",
    "AMT_GOODS_PRICE",
    "NAME_INCOME_TYPE",
    "NAME_EDUCATION_TYPE",
    "NAME_FAMILY_STATUS",
    "NAME_HOUSING_TYPE",
    "OCCUPATION_TYPE",
    "ORGANIZATION_TYPE",
    "DAYS_BIRTH",
    "DAYS_EMPLOYED",
    "DAYS_REGISTRATION",
    "DAYS_ID_PUBLISH",
    "REGION_POPULATION_RELATIVE",
    "REGION_RATING_CLIENT",
    "REGION_RATING_CLIENT_W_CITY",
    "EXT_SOURCE_1",
    "EXT_SOURCE_2",
    "EXT_SOURCE_3",
    "AMT_REQ_CREDIT_BUREAU_HOUR",
    "AMT_REQ_CREDIT_BUREAU_DAY",
    "AMT_REQ_CREDIT_BUREAU_WEEK",
    "AMT_REQ_CREDIT_BUREAU_MON",
    "AMT_REQ_CREDIT_BUREAU_QRT",
    "AMT_REQ_CREDIT_BUREAU_YEAR",
]

missing_initial_features = [
    col for col in initial_features
    if col not in X_train.columns
]

if missing_initial_features:
    raise ValueError(
        f"Initial features missing from dataset: {missing_initial_features}"
    )

X_train_fe = X_train[initial_features].copy()
X_test_fe = X_test[initial_features].copy()

print("Initial feature count:", len(initial_features))
print("Training feature dataset:", X_train_fe.shape)
print("Test feature dataset:", X_test_fe.shape)

Initial feature count: 32
Training feature dataset: (246008, 32)
Test feature dataset: (61503, 32)


In [105]:
feature_summary = pd.DataFrame({
    "dtype": X_train_fe.dtypes.astype(str),
    "missing_count": X_train_fe.isna().sum(),
    "missing_rate": X_train_fe.isna().mean(),
    "n_unique": X_train_fe.nunique(dropna=False),
}).sort_values("missing_rate", ascending=False)

feature_summary.head(20)

,dtype,missing_count,missing_rate,n_unique
EXT_SOURCE_1,float64,138595,0.563376,94565
OCCUPATION_TYPE,str,76940,0.312754,19
EXT_SOURCE_3,float64,48805,0.198388,807
AMT_REQ_CREDIT_BUREAU_YEAR,float64,33244,0.135134,25
AMT_REQ_CREDIT_BUREAU_QRT,float64,33244,0.135134,11
AMT_REQ_CREDIT_BUREAU_MON,float64,33244,0.135134,24
AMT_REQ_CREDIT_BUREAU_WEEK,float64,33244,0.135134,10
AMT_REQ_CREDIT_BUREAU_DAY,float64,33244,0.135134,10
AMT_REQ_CREDIT_BUREAU_HOUR,float64,33244,0.135134,6
EXT_SOURCE_2,float64,531,0.002158,108826


## 5. Reusable Domain Feature Engineering

In [106]:
def safe_divide(
    numerator: pd.Series,
    denominator: pd.Series
) -> pd.Series:
    """Safely divide two pandas Series."""
    denominator_clean = denominator.replace(0, np.nan)
    result = numerator / denominator_clean
    return result.replace([np.inf, -np.inf], np.nan)

In [107]:
def create_domain_features(data: pd.DataFrame) -> pd.DataFrame:
    """
    Create interpretable credit-risk features from raw application data.

    This function includes all deterministic cleaning required before the
    learned sklearn preprocessing pipeline is applied.
    """
    data = data.copy()

    # --------------------------------------------------
    # Special-value cleaning
    # --------------------------------------------------
    special_employed_value = 365243

    data["DAYS_EMPLOYED_SPECIAL_FLAG"] = (
        data["DAYS_EMPLOYED"] == special_employed_value
    ).astype("int8")

    data["DAYS_EMPLOYED"] = data["DAYS_EMPLOYED"].replace(
        special_employed_value,
        np.nan
    )

    # --------------------------------------------------
    # Applicant age and employment history
    # --------------------------------------------------
    data["AGE_YEARS"] = -data["DAYS_BIRTH"] / 365.25
    data["EMPLOYMENT_YEARS"] = -data["DAYS_EMPLOYED"] / 365.25
    data["REGISTRATION_YEARS"] = -data["DAYS_REGISTRATION"] / 365.25
    data["ID_PUBLISH_YEARS"] = -data["DAYS_ID_PUBLISH"] / 365.25

    data.loc[
        (data["AGE_YEARS"] < 18) | (data["AGE_YEARS"] > 100),
        "AGE_YEARS"
    ] = np.nan

    data.loc[
        data["EMPLOYMENT_YEARS"] < 0,
        "EMPLOYMENT_YEARS"
    ] = np.nan

    data.loc[
        data["EMPLOYMENT_YEARS"] > data["AGE_YEARS"],
        "EMPLOYMENT_YEARS"
    ] = np.nan

    # --------------------------------------------------
    # Affordability features
    # --------------------------------------------------
    data["CREDIT_INCOME_RATIO"] = safe_divide(
        data["AMT_CREDIT"],
        data["AMT_INCOME_TOTAL"]
    )

    data["ANNUITY_INCOME_RATIO"] = safe_divide(
        data["AMT_ANNUITY"],
        data["AMT_INCOME_TOTAL"]
    )

    data["CREDIT_ANNUITY_RATIO"] = safe_divide(
        data["AMT_CREDIT"],
        data["AMT_ANNUITY"]
    )

    data["INCOME_PER_PERSON"] = safe_divide(
        data["AMT_INCOME_TOTAL"],
        data["CNT_FAM_MEMBERS"]
    )

    data["CREDIT_PER_PERSON"] = safe_divide(
        data["AMT_CREDIT"],
        data["CNT_FAM_MEMBERS"]
    )

    # --------------------------------------------------
    # Loan structure features
    # --------------------------------------------------
    data["GOODS_CREDIT_RATIO"] = safe_divide(
        data["AMT_GOODS_PRICE"],
        data["AMT_CREDIT"]
    )

    data["CREDIT_GOODS_DIFFERENCE"] = (
        data["AMT_CREDIT"] - data["AMT_GOODS_PRICE"]
    )

    data["ANNUITY_GOODS_RATIO"] = safe_divide(
        data["AMT_ANNUITY"],
        data["AMT_GOODS_PRICE"]
    )

    # --------------------------------------------------
    # Employment stability
    # --------------------------------------------------
    data["EMPLOYMENT_AGE_RATIO"] = safe_divide(
        data["EMPLOYMENT_YEARS"],
        data["AGE_YEARS"]
    )

    # --------------------------------------------------
    # External scores
    # --------------------------------------------------
    ext_source_cols = [
        "EXT_SOURCE_1",
        "EXT_SOURCE_2",
        "EXT_SOURCE_3",
    ]

    data["EXT_SOURCE_MEAN"] = data[ext_source_cols].mean(axis=1)
    data["EXT_SOURCE_MIN"] = data[ext_source_cols].min(axis=1)
    data["EXT_SOURCE_MAX"] = data[ext_source_cols].max(axis=1)
    data["EXT_SOURCE_STD"] = data[ext_source_cols].std(axis=1)

    data["EXT_SOURCE_MISSING_COUNT"] = (
        data[ext_source_cols]
        .isna()
        .sum(axis=1)
        .astype("int8")
    )

    # --------------------------------------------------
    # Bureau inquiries
    # --------------------------------------------------
    bureau_inquiry_cols = [
        "AMT_REQ_CREDIT_BUREAU_HOUR",
        "AMT_REQ_CREDIT_BUREAU_DAY",
        "AMT_REQ_CREDIT_BUREAU_WEEK",
        "AMT_REQ_CREDIT_BUREAU_MON",
        "AMT_REQ_CREDIT_BUREAU_QRT",
        "AMT_REQ_CREDIT_BUREAU_YEAR",
    ]

    data["BUREAU_INQUIRY_TOTAL"] = (
        data[bureau_inquiry_cols]
        .sum(axis=1, min_count=1)
    )

    recent_inquiry_cols = [
        "AMT_REQ_CREDIT_BUREAU_HOUR",
        "AMT_REQ_CREDIT_BUREAU_DAY",
        "AMT_REQ_CREDIT_BUREAU_WEEK",
        "AMT_REQ_CREDIT_BUREAU_MON",
    ]

    data["BUREAU_INQUIRY_RECENT"] = (
        data[recent_inquiry_cols]
        .sum(axis=1, min_count=1)
    )

    # --------------------------------------------------
    # Remove exact linear duplicates of year features
    # --------------------------------------------------
    raw_time_features_to_drop = [
        "DAYS_BIRTH",
        "DAYS_EMPLOYED",
        "DAYS_REGISTRATION",
        "DAYS_ID_PUBLISH",
    ]

    data = data.drop(columns=raw_time_features_to_drop)

    # --------------------------------------------------
    # Applicant-level missingness
    # --------------------------------------------------
    data["TOTAL_MISSING_COUNT"] = (
        data.isna()
        .sum(axis=1)
        .astype("int16")
    )

    data["TOTAL_MISSING_RATE"] = data.isna().mean(axis=1)

    return data

In [108]:
X_train_fe = create_domain_features(X_train_fe)
X_test_fe = create_domain_features(X_test_fe)

print("Training shape after feature engineering:", X_train_fe.shape)
print("Test shape after feature engineering:", X_test_fe.shape)

Training shape after feature engineering: (246008, 51)
Test shape after feature engineering: (61503, 51)


## 6. Feature Engineering Validation

In [109]:
assert list(X_train_fe.columns) == list(X_test_fe.columns)

numeric_train = X_train_fe.select_dtypes(include=["number"])
numeric_test = X_test_fe.select_dtypes(include=["number"])

train_infinite_count = np.isinf(numeric_train).sum().sum()
test_infinite_count = np.isinf(numeric_test).sum().sum()

print("Infinite values in training data:", train_infinite_count)
print("Infinite values in test data:", test_infinite_count)

assert train_infinite_count == 0
assert test_infinite_count == 0

Infinite values in training data: 0
Infinite values in test data: 0


In [110]:
validation_summary = pd.DataFrame({
    "dtype": X_train_fe.dtypes.astype(str),
    "missing_count": X_train_fe.isna().sum(),
    "missing_rate": X_train_fe.isna().mean(),
    "n_unique": X_train_fe.nunique(dropna=False),
}).sort_values("missing_rate", ascending=False)

validation_summary.head(25)

,dtype,missing_count,missing_rate,n_unique
EXT_SOURCE_1,float64,138595,0.563376,94565
OCCUPATION_TYPE,str,76940,0.312754,19
EXT_SOURCE_3,float64,48805,0.198388,807
EMPLOYMENT_YEARS,float64,44143,0.179437,12104
EMPLOYMENT_AGE_RATIO,float64,44143,0.179437,200542
AMT_REQ_CREDIT_BUREAU_DAY,float64,33244,0.135134,10
AMT_REQ_CREDIT_BUREAU_HOUR,float64,33244,0.135134,6
AMT_REQ_CREDIT_BUREAU_MON,float64,33244,0.135134,24
AMT_REQ_CREDIT_BUREAU_QRT,float64,33244,0.135134,11
AMT_REQ_CREDIT_BUREAU_YEAR,float64,33244,0.135134,25


In [111]:
engineered_features = [
    "AGE_YEARS",
    "EMPLOYMENT_YEARS",
    "REGISTRATION_YEARS",
    "ID_PUBLISH_YEARS",
    "CREDIT_INCOME_RATIO",
    "ANNUITY_INCOME_RATIO",
    "CREDIT_ANNUITY_RATIO",
    "INCOME_PER_PERSON",
    "CREDIT_PER_PERSON",
    "GOODS_CREDIT_RATIO",
    "CREDIT_GOODS_DIFFERENCE",
    "ANNUITY_GOODS_RATIO",
    "EMPLOYMENT_AGE_RATIO",
    "EXT_SOURCE_MEAN",
    "EXT_SOURCE_MIN",
    "EXT_SOURCE_MAX",
    "EXT_SOURCE_STD",
    "EXT_SOURCE_MISSING_COUNT",
    "BUREAU_INQUIRY_TOTAL",
    "BUREAU_INQUIRY_RECENT",
    "TOTAL_MISSING_COUNT",
    "TOTAL_MISSING_RATE",
]

X_train_fe[engineered_features].describe().T[
    ["count", "mean", "std", "min", "50%", "max"]
].round(3)

,count,mean,std,min,50%,max
AGE_YEARS,246008.0,43.886,11.945,20.504,43.105,6.907300e+01
EMPLOYMENT_YEARS,201865.0,6.530,6.406,-0.000,4.512,4.904000e+01
REGISTRATION_YEARS,246008.0,13.661,9.650,-0.000,12.334,6.754800e+01
ID_PUBLISH_YEARS,246008.0,8.198,4.129,0.000,8.912,1.970400e+01
CREDIT_INCOME_RATIO,246008.0,3.959,2.688,0.005,3.269,4.922700e+01
ANNUITY_INCOME_RATIO,245998.0,0.181,0.095,0.000,0.163,1.571000e+00
CREDIT_ANNUITY_RATIO,245998.0,21.624,7.821,8.037,20.000,4.530500e+01
INCOME_PER_PERSON,246006.0,93113.103,106834.976,3375.000,75000.000,3.900000e+07
CREDIT_PER_PERSON,246006.0,323994.012,258839.896,6750.000,256032.000,4.031032e+06
GOODS_CREDIT_RATIO,245787.0,0.901,0.097,0.167,0.894,6.667000e+00


## 7. Descriptive Target Comparison

In [112]:
train_feature_review = X_train_fe.copy()
train_feature_review["TARGET"] = y_train

feature_target_review = (
    train_feature_review
    .groupby("TARGET")[engineered_features]
    .mean()
    .T
)

feature_target_review.columns = [
    "TARGET_0_MEAN",
    "TARGET_1_MEAN",
]

feature_target_review["ABSOLUTE_DIFFERENCE"] = (
    feature_target_review["TARGET_1_MEAN"]
    - feature_target_review["TARGET_0_MEAN"]
)

feature_target_review.sort_values(
    by="ABSOLUTE_DIFFERENCE",
    key=lambda series: series.abs(),
    ascending=False
)

,TARGET_0_MEAN,TARGET_1_MEAN,ABSOLUTE_DIFFERENCE
CREDIT_PER_PERSON,325815.426935,303253.548051,-22561.878883
CREDIT_GOODS_DIFFERENCE,60286.350945,68836.601144,8550.250199
INCOME_PER_PERSON,93283.377693,91174.187608,-2109.190086
AGE_YEARS,44.167717,40.683288,-3.484430
EMPLOYMENT_YEARS,6.678030,4.966719,-1.711310
REGISTRATION_YEARS,13.781205,12.288987,-1.492218
CREDIT_ANNUITY_RATIO,21.701282,20.747626,-0.953655
ID_PUBLISH_YEARS,8.258212,7.506630,-0.751582
TOTAL_MISSING_COUNT,2.623189,2.852064,0.228875
EXT_SOURCE_MIN,0.409760,0.282303,-0.127458


## 8. Feature-Type Definition

In [113]:
categorical_features = (
    X_train_fe
    .select_dtypes(include=["object", "string", "category"])
    .columns
    .tolist()
)

numerical_features = (
    X_train_fe
    .select_dtypes(include=["number"])
    .columns
    .tolist()
)

classified_features = set(
    categorical_features + numerical_features
)

unclassified_features = (
    set(X_train_fe.columns) - classified_features
)

print("Number of categorical features:", len(categorical_features))
print("Number of numerical features:", len(numerical_features))
print("Unclassified features:", unclassified_features)

assert not unclassified_features

Number of categorical features: 10
Number of numerical features: 41
Unclassified features: set()


In [114]:
categorical_cardinality = (
    X_train_fe[categorical_features]
    .nunique(dropna=False)
    .sort_values(ascending=False)
    .to_frame("n_unique")
)

categorical_cardinality

,n_unique
ORGANIZATION_TYPE,58
OCCUPATION_TYPE,19
NAME_INCOME_TYPE,8
NAME_FAMILY_STATUS,6
NAME_HOUSING_TYPE,6
NAME_EDUCATION_TYPE,5
CODE_GENDER,3
NAME_CONTRACT_TYPE,2
FLAG_OWN_CAR,2
FLAG_OWN_REALTY,2


### High-Cardinality Note

`ORGANIZATION_TYPE` may exceed the initial high-cardinality threshold, but it is retained for the baseline because the final encoded feature space remains manageable. Rare-category grouping can be evaluated later if it improves stability or interpretability.

## 9. Preprocessing Pipeline

In [115]:
numerical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                add_indicator=True
            )
        ),
        (
            "scaler",
            StandardScaler()
        ),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True
            )
        ),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numerical",
            numerical_pipeline,
            numerical_features
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features
        ),
    ],
    remainder="drop",
    verbose_feature_names_out=True
)

In [116]:
X_train_processed = preprocessor.fit_transform(X_train_fe)
X_test_processed = preprocessor.transform(X_test_fe)

print("Processed training shape:", X_train_processed.shape)
print("Processed test shape:", X_test_processed.shape)

assert X_train_processed.shape[1] == X_test_processed.shape[1]
assert X_train_processed.shape[0] == len(y_train)
assert X_test_processed.shape[0] == len(y_test)

Processed training shape: (246008, 178)
Processed test shape: (61503, 178)


In [117]:
processed_feature_names = preprocessor.get_feature_names_out()

clean_feature_names = [
    feature_name
    .replace("numerical__", "")
    .replace("categorical__", "")
    for feature_name in processed_feature_names
]

print("Number of processed features:", len(clean_feature_names))
clean_feature_names[:30]

Number of processed features: 178


['CNT_CHILDREN',
 'CNT_FAM_MEMBERS',
 'AMT_INCOME_TOTAL',
 'AMT_CREDIT',
 'AMT_ANNUITY',
 'AMT_GOODS_PRICE',
 'REGION_POPULATION_RELATIVE',
 'REGION_RATING_CLIENT',
 'REGION_RATING_CLIENT_W_CITY',
 'EXT_SOURCE_1',
 'EXT_SOURCE_2',
 'EXT_SOURCE_3',
 'AMT_REQ_CREDIT_BUREAU_HOUR',
 'AMT_REQ_CREDIT_BUREAU_DAY',
 'AMT_REQ_CREDIT_BUREAU_WEEK',
 'AMT_REQ_CREDIT_BUREAU_MON',
 'AMT_REQ_CREDIT_BUREAU_QRT',
 'AMT_REQ_CREDIT_BUREAU_YEAR',
 'DAYS_EMPLOYED_SPECIAL_FLAG',
 'AGE_YEARS',
 'EMPLOYMENT_YEARS',
 'REGISTRATION_YEARS',
 'ID_PUBLISH_YEARS',
 'CREDIT_INCOME_RATIO',
 'ANNUITY_INCOME_RATIO',
 'CREDIT_ANNUITY_RATIO',
 'INCOME_PER_PERSON',
 'CREDIT_PER_PERSON',
 'GOODS_CREDIT_RATIO',
 'CREDIT_GOODS_DIFFERENCE']

## 10. Transformation Validation

In [118]:
def count_missing_and_infinite(matrix):
    values = matrix.data if hasattr(matrix, "data") else matrix
    return {
        "missing": int(np.isnan(values).sum()),
        "infinite": int(np.isinf(values).sum()),
    }

train_validation = count_missing_and_infinite(X_train_processed)
test_validation = count_missing_and_infinite(X_test_processed)

print("Training:", train_validation)
print("Test:", test_validation)

assert train_validation["missing"] == 0
assert test_validation["missing"] == 0
assert train_validation["infinite"] == 0
assert test_validation["infinite"] == 0

Training: {'missing': 0, 'infinite': 0}
Test: {'missing': 0, 'infinite': 0}


## 11. Save Preprocessing Artifacts

The saved sklearn preprocessor expects data after `create_domain_features()` has been applied.

Correct usage for new raw application data:

```python
new_data_fe = create_domain_features(new_data)
new_data_processed = preprocessor.transform(new_data_fe)
```

The deterministic feature-engineering function and the learned sklearn preprocessor must both be preserved for future scoring.

In [119]:
PREPROCESSOR_PATH = MODEL_DIR / "feature_preprocessor.joblib"
FEATURE_NAMES_PATH = MODEL_DIR / "processed_feature_names.txt"

joblib.dump(preprocessor, PREPROCESSOR_PATH)

with open(FEATURE_NAMES_PATH, "w", encoding="utf-8") as file:
    for feature_name in clean_feature_names:
        file.write(f"{feature_name}\n")

print("Saved preprocessor to:", PREPROCESSOR_PATH)
print("Saved feature names to:", FEATURE_NAMES_PATH)

Saved preprocessor to: /Users/hxxy/Desktop/找工/credit-risk-decisioning/models/feature_preprocessor.joblib
Saved feature names to: /Users/hxxy/Desktop/找工/credit-risk-decisioning/models/processed_feature_names.txt


In [120]:
train_split_output = pd.DataFrame({
    "SK_ID_CURR": train_ids,
    "TARGET": y_train,
}).reset_index(drop=True)

test_split_output = pd.DataFrame({
    "SK_ID_CURR": test_ids,
    "TARGET": y_test,
}).reset_index(drop=True)

TRAIN_SPLIT_PATH = (
    PROCESSED_DATA_DIR / "train_split_ids_targets.csv"
)

TEST_SPLIT_PATH = (
    PROCESSED_DATA_DIR / "test_split_ids_targets.csv"
)

train_split_output.to_csv(TRAIN_SPLIT_PATH, index=False)
test_split_output.to_csv(TEST_SPLIT_PATH, index=False)

print("Saved:", TRAIN_SPLIT_PATH)
print("Saved:", TEST_SPLIT_PATH)

Saved: /Users/hxxy/Desktop/找工/credit-risk-decisioning/data/processed/train_split_ids_targets.csv
Saved: /Users/hxxy/Desktop/找工/credit-risk-decisioning/data/processed/test_split_ids_targets.csv


## 12. Final Summary

In [121]:
preprocessing_summary = pd.Series({
    "training_rows": X_train_fe.shape[0],
    "test_rows": X_test_fe.shape[0],
    "raw_feature_count_after_engineering": X_train_fe.shape[1],
    "categorical_feature_count": len(categorical_features),
    "numerical_feature_count": len(numerical_features),
    "processed_feature_count": X_train_processed.shape[1],
    "training_target_rate": y_train.mean(),
    "test_target_rate": y_test.mean(),
    "training_missing_after_processing": train_validation["missing"],
    "test_missing_after_processing": test_validation["missing"],
    "training_infinite_after_processing": train_validation["infinite"],
    "test_infinite_after_processing": test_validation["infinite"],
})

preprocessing_summary

training_rows                          246008.000000
test_rows                               61503.000000
raw_feature_count_after_engineering        51.000000
categorical_feature_count                  10.000000
numerical_feature_count                    41.000000
processed_feature_count                   178.000000
training_target_rate                        0.080729
test_target_rate                            0.080728
training_missing_after_processing           0.000000
test_missing_after_processing               0.000000
training_infinite_after_processing          0.000000
test_infinite_after_processing              0.000000
dtype: float64

## Preprocessing Summary

The workflow now:

- performs a stratified train/test split before learning preprocessing parameters,
- removes the applicant identifier from model features,
- handles `DAYS_EMPLOYED = 365243` inside the reusable feature function,
- creates interpretable credit-risk and affordability features,
- removes exact linear duplicates of converted time variables,
- preserves informative missingness,
- uses median imputation and missing indicators for numerical variables,
- uses most-frequent imputation and one-hot encoding for categorical variables,
- safely handles unseen categories,
- confirms that no missing or infinite values remain,
- saves the fitted preprocessor and feature names.

The next notebook can build and evaluate an interpretable logistic regression baseline.